In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pylab as plt
import cv2
import os
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [ ]:
path = "labeled_data.csv"
df = pd.read_csv(path)
data = df.copy()
data.head()

,uid,MeSH,findings,impression,filename,label
0,1,normal,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,1_IM-0001-4001.dcm.png,normal
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.,2_IM-0652-1001.dcm.png,abnormal
2,4,"Pulmonary Disease, Chronic Obstructive;Bullous...",There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...,4_IM-2050-1001.dcm.png,abnormal
3,5,Osteophyte/thoracic vertebrae/multiple/small;T...,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.,5_IM-2117-1003002.dcm.png,abnormal
4,6,normal,Heart size and mediastinal contour are within ...,No acute cardiopulmonary findings.,6_IM-2192-1001.dcm.png,normal


In [ ]:
data['label'] = data['label'].map({'normal' : 0, 'abnormal' : 1})

In [ ]:
data.head()

,uid,MeSH,findings,impression,filename,label
0,1,normal,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,1_IM-0001-4001.dcm.png,0
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.,2_IM-0652-1001.dcm.png,1
2,4,"Pulmonary Disease, Chronic Obstructive;Bullous...",There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...,4_IM-2050-1001.dcm.png,1
3,5,Osteophyte/thoracic vertebrae/multiple/small;T...,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.,5_IM-2117-1003002.dcm.png,1
4,6,normal,Heart size and mediastinal contour are within ...,No acute cardiopulmonary findings.,6_IM-2192-1001.dcm.png,0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
filepath = "/content/drive/MyDrive/required_images"

In [ ]:
def train_transform_pipeline(filepath):
    transform = transforms.Compose([
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomResizedCrop(size=224, scale=(0.9, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    img = cv2.imread(filepath, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = Image.fromarray(img)
    img = transform(img)
    return img

In [ ]:
def transform_pipeline(filepath):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    img = cv2.imread(filepath, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = transform(img)
    return img

In [ ]:
import torch

In [ ]:
class ChestXrayDataset(torch.utils.data.Dataset):
      def __init__(self,df,filepath,transform_fn):
         self.df = df.reset_index(drop=True)
         self.filepath = filepath
         self.transform_fn = transform_fn

      def __len__(self):
         return len(self.df)

      def __getitem__(self, index):
          img_path = os.path.join(self.filepath,self.df.loc[index,'filename'])
          img = self.transform_fn(img_path)
          label = self.df.loc[index,'label']
          return img, label

In [ ]:
train_df,temp_df = train_test_split(
    data,
    test_size = 0.2,
    random_state = 42,
    stratify = data['label']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size = 0.5,
    random_state = 42,
    stratify = temp_df['label']
)

In [ ]:
train_dataset = ChestXrayDataset(train_df,filepath,train_transform_pipeline)
val_dataset = ChestXrayDataset(val_df,filepath,transform_pipeline)
test_dataset = ChestXrayDataset(test_df,filepath,transform_pipeline)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
train_loader = DataLoader(dataset = train_dataset,batch_size=16,shuffle=True,num_workers=2)
val_loader = DataLoader(dataset = val_dataset,batch_size=16,shuffle=False,num_workers=2)
test_loader = DataLoader(dataset = test_dataset,batch_size=16,shuffle=False,num_workers=2)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights = ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s]


In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
import torch.nn as nn

In [ ]:
model.fc = nn.Linear(model.fc.in_features,2)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
import torch.optim as optim

In [ ]:
optimizer = optim.Adam(model.parameters(),lr = 0.001)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = model.to(device)

In [ ]:
for epoch in range(10):
    loss_train_list = list()
    loss_val_list = list()
    model.train()
    for batch_idx , (image , label) in enumerate(train_loader):
        optimizer.zero_grad()
        image = image.to(device)
        outputs = model(image)
        label = label.to(device)
        loss_train = criterion(outputs,label)
        loss_train.backward()
        loss_train_list.append(loss_train.item())
        optimizer.step()
    torch.save(model.state_dict(),f"/content/drive/MyDrive/models/epoch_{epoch + 1}.pt")
    model.eval()
    for batch_idx , (image , label) in enumerate(val_loader):
        with torch.no_grad():
             image = image.to(device)
             outputs = model(image)
             label = label.to(device)
             loss_val = criterion(outputs,label)
             loss_val_list.append(loss_val.item())
    mean_train = sum(loss_train_list)/len(loss_train_list)
    mean_val = sum(loss_val_list)/len(loss_val_list)
    print(f"Epoch : {epoch + 1}, Train Loss : {mean_train:.4f}, Val Loss : {mean_val:.4f}")

Epoch : 1, Train Loss : 0.5994, Val Loss : 0.5838
Epoch : 2, Train Loss : 0.5972, Val Loss : 0.6434
Epoch : 3, Train Loss : 0.6006, Val Loss : 0.6058
Epoch : 4, Train Loss : 0.5894, Val Loss : 0.5691
Epoch : 5, Train Loss : 0.6077, Val Loss : 0.5697
Epoch : 6, Train Loss : 0.5817, Val Loss : 0.6116
Epoch : 7, Train Loss : 0.5983, Val Loss : 0.6135
Epoch : 8, Train Loss : 0.5880, Val Loss : 0.5909
Epoch : 9, Train Loss : 0.5851, Val Loss : 0.5753
Epoch : 10, Train Loss : 0.5929, Val Loss : 0.5820
